In [20]:
import pandas as pd
import datetime
from dateutil import parser as dtparser
from IPython.display import display as DisplayHandle, clear_output

### Importing the modules
import tomtom_api
import processor
import transformer
import csv_handler

In [22]:
# Define a TomTom API Client
ttapi = tomtom_api.Client(api_key = "SdbkkAPVV6GxzS7beuYj8mqYnSRWgUmx")

# Define a tomtom processor
tt_processor = processor.Processor(ttapi)
transformer = transformer.Transformer()
csv_handler = csv_handler.CSVHandler()

In [15]:
# Starting Point and Ending Point of Route Analysis
centroids = csv_handler.readCentroids()
from_cluster = 0
to_cluster = 1
start_point = f"{centroids['Latitudine'].iloc[from_cluster]},{centroids['Longitudine'].iloc[from_cluster]}"              
end_point = f"{centroids['Latitudine'].iloc[to_cluster]},{centroids['Longitudine'].iloc[to_cluster]}"
print (f"Start Point: {start_point}, End Point: {end_point}")

Start Point: 40.592006479897954,17.1154860457833, End Point: 40.59368321711289,17.111410033960354


In [16]:
# Date Range for Route Analysis
year = 2024
month = 7
day_range = range(1, 31)

# define time slots in a day
# slots = ["00:00-06:00", "06:00-12:00", "12:00-18:00", "18:00-23:59"]
slots = ["07:00-10:00", "13:00-15:00", "18:00-21:00"]

In [17]:
# Get the route summaries for each day in the date range
dd1 = DisplayHandle("Processing historing summaries...", display_id=True)
for day in day_range:
    date = datetime.datetime(year, month, day)
    route_summaries = tt_processor.getHistoricData(start_point, end_point, date)
    csv_handler.writeHistoricData(route_summaries, date)
    dd1.update(f"Processing historic summaries { day / len(day_range) * 100 }% ...")
dd1.update("Historic summaries processed.")

Retrieved data: 100% >> 2024-07-30 23:30:00


In [23]:
# Calculate the traffic data for each day in the date range
dd = DisplayHandle("Calculating traffic data...", display_id=True)
for day in day_range:
    date = datetime.datetime(year, month, day)
    historic_data = csv_handler.readHistoricData(date)
    traffic_data = transformer.calculateTrafficData(historic_data)
    csv_handler.writeTrafficData(traffic_data, date)
    dd.update(f"Calculating traffic data { round(day / len(day_range) * 100) }% ...")
dd.update("Traffic data calculated.")

Computing traffic: 100% >> 2024-07-30T23:30:00+02:00


In [24]:
for day in day_range:
    date = datetime.datetime(year, month, day)
    traffic_data = csv_handler.readTrafficData(date)
    slotted_traffic = transformer.calculateSlottedTraffic(traffic_data, slots)
    csv_handler.writeSlottedTrafficData(slotted_traffic, date)
print ("Done")

Done


In [25]:
combined_data = csv_handler.readAllSlotted()
traffic = transformer.getAggregatedTrafficData(combined_data, slots)

In [26]:
traffic

,slot,avg_traffic_speed,avg_density_factor,avg_number_of_vehicles,points
0,07:00-10:00,15.554335,1.204004,149.827778,"[{'latitude': 40.59197, 'longitude': 17.11565}..."
1,13:00-15:00,15.831475,1.178829,146.750000,"[{'latitude': 40.59197, 'longitude': 17.11565}..."
2,18:00-21:00,13.135251,1.428979,178.016667,"[{'latitude': 40.59197, 'longitude': 17.11565}..."
